In [ ]:
# UPAR27 Track3 v5 LARGE | aashish | ViTPose-L (78.3 AP, +4.5 over Small), FULL real data, heavy aug, val-selected bestHF_TOKEN = os.environ.get("HF_TOKEN", "hf_YOUR_TOKEN_HERE")  # was hardcoded during Kaggle runs; set HF_TOKEN env/secretHF_REPO="ShivRamSaud/upar-track3-pose-large"MID="usyd-community/vitpose-plus-large"import os, time, jsonos.environ['PYTORCH_CUDA_ALLOC_CONF']='expandable_segments:True'

In [ ]:
!pip install -q transformers huggingface_hub opencv-python pycocotools
import torch
print(torch.__version__, torch.cuda.is_available(), torch.cuda.device_count())
for i in range(torch.cuda.device_count()): print(i, torch.cuda.get_device_name(i), torch.cuda.get_device_properties(i).total_memory/1e9)

In [ ]:
from huggingface_hub import login, create_repo
login(token=HF_TOKEN)
create_repo(HF_REPO, exist_ok=True)
print('HF', HF_REPO)

In [ ]:
!rm -rf /tmp/t3 && git clone --depth 1 https://github.com/MickaelCormier/UPAR-Challenge-2027-Track3 /tmp/t3
!ls /kaggle/input 2>&1 | head -20

In [ ]:
# Load ViTPose-L (COCO MoE head idx 0; compliant pretraining)
from transformers import VitPoseForPoseEstimation, VitPoseImageProcessor
proc = VitPoseImageProcessor.from_pretrained(MID)
model = VitPoseForPoseEstimation.from_pretrained(MID)
print('labels', model.config.num_labels, 'scale', model.config.scale_factor)
print('processor size', proc.size)
model = model.cuda().train()
try:
 model.vision_model.encoder.gradient_checkpointing_enable()
except Exception as e: print('gradckpt', e)
print('params M', sum(p.numel() for p in model.parameters())/1e6)

In [ ]:
Data cell: basename index (157927 imgs), train_upar_all.json (42826 anns), resolve_img, 384x288 probe w/ 256x192 fallback, MEAN/STD, PoseDS(train flag, scale-jitter, blur, color, random-erase; NO flip/rotation for label safety), gaussian heatmaps (17 joints, ears masked), 95/5 split. Kept ALL real images.

In [ ]:
Train cell: resume best/step from HF, BS8/ACC4 (384) or BS16 (256), AdamW 5e-5 backbone / 2e-4 head, OneCycle, AMP, val-MSE per epoch, best_val to best_model.bin + last_model.bin, HF push per epoch, 6.2h guard. Curve ep0 val 0.00082 -> ep6 best 0.00047 (19 epochs).

In [ ]:
# Export BEST (not last) exactly like official export_model.py
import json as js
torch.set_grad_enabled(False)
model.load_state_dict(torch.load('/tmp/hf3L/best_model.bin', map_location='cpu'))
model.eval()
H,W=HW
class ExpW(torch.nn.Module):
 def __init__(self,m): super().__init__(); self.m=m; self.register_buffer('di',torch.tensor([0]))
 def forward(self,pv): return self.m(pixel_values=pv,dataset_index=self.di.expand(pv.shape[0])).heatmaps
ex=torch.zeros(1,3,H,W)
trm=torch.jit.trace(ExpW(model).eval(),ex,strict=False)
trm=torch.jit.freeze(trm); trm.save('/tmp/vitpose.torchscript')
print('traced')
meta={'source_checkpoint':MID,'dataset_index':0,'input_height':H,'input_width':W,'image_mean':list(proc.image_mean),'image_std':list(proc.image_std),'rescale_factor':float(proc.rescale_factor),'normalize_factor':float(getattr(proc,'normalize_factor',200.0)),'padding_factor':1.25,'num_keypoints':int(model.config.num_labels),'heatmap_scale_factor':int(model.config.scale_factor)}
open('/tmp/preprocess.json','w').write(js.dumps(meta,indent=2))
from huggingface_hub import HfApi
HfApi().upload_file(repo_id=HF_REPO,path_in_repo='vitpose.torchscript',path_or_fileobj='/tmp/vitpose.torchscript',commit_message='ts-best')
HfApi().upload_file(repo_id=HF_REPO,path_in_repo='preprocess.json',path_or_fileobj='/tmp/preprocess.json',commit_message='meta')
print('uploaded BEST TS + meta to', HF_REPO)